# Sales Prediction Using Python
OIBSIP · Data Science · Task 5

**Goal:** predict product sales from advertising spend on TV, Radio and Newspaper.

**Dataset:** `Advertising.csv` (download from Kaggle: search *advertising sales prediction* and place the file next to this notebook).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load data and EDA

In [ ]:
df = pd.read_csv("Advertising.csv")
# Some versions of the file have an unnamed index column
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
print("Shape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
df.head()

In [ ]:
df.describe()

In [ ]:
sns.pairplot(df)
plt.show()

### Sales versus each advertising channel

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["TV", "Radio", "Newspaper"]):
    sns.scatterplot(data=df, x=col, y="Sales", ax=ax)
    sns.regplot(data=df, x=col, y="Sales", scatter=False, color="red", ax=ax)
    ax.set_title(f"Sales vs {col}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.show()

**Observation:** TV has the strongest correlation with Sales, Radio is moderate, and Newspaper is weak. Write your own observations here after viewing the charts.

## 2. Train / test split

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Baseline: Linear Regression, then Random Forest

In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}, pred

lin = LinearRegression()
rf = RandomForestRegressor(n_estimators=200, random_state=42)
res_lin, pred_lin = evaluate("Linear Regression", lin)
res_rf, pred_rf = evaluate("Random Forest", rf)
results = pd.DataFrame([res_lin, res_rf]).set_index("Model").round(3)
results

## 4. Residual plot of the best model

In [ ]:
best_name = results["R2"].idxmax()
best_pred = pred_lin if best_name == "Linear Regression" else pred_rf
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(best_pred, residuals)
axes[0].axhline(0, color="red")
axes[0].set_xlabel("Predicted sales"); axes[0].set_ylabel("Residual")
axes[0].set_title(f"Residuals - {best_name}")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.show()
print("Best model:", best_name)

If residuals scatter randomly around zero the model is capturing the pattern; a visible curve or funnel shape means errors are systematic. Describe what you see.

## 5. Which channel matters most?

In [ ]:
coef = pd.Series(lin.coef_, index=X.columns, name="Linear coefficient")
importance = pd.Series(rf.feature_importances_, index=X.columns, name="RF importance")
print(pd.concat([coef, importance], axis=1).round(3))

importance.sort_values().plot(kind="barh", title="Random Forest feature importance")
plt.show()

## 6. Conclusion
- TV advertising has the largest effect on sales, followed by Radio.
- Newspaper spend has a small coefficient/importance, so extra newspaper budget is unlikely to pay off.
- Recommendation: shift budget toward TV and Radio. (Edit this section to match your own results.)